# 🏗️ DeepSeek V3 전체 모델 구조와 추론 흐름

이 노트북에서는 DeepSeek V3의 **전체 모델 구조**를 살펴보고, **추론 과정**에서 데이터가 어떻게 흐르는지 학습합니다.

**참고 자료:**
- 논문: https://arxiv.org/pdf/2412.19437

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


## 1. DeepSeek V3 설정

### 주요 스펙
| 구성 요소 | 값 | 설명 |
|-----------|-----|------|
| vocab_size | 102,400 | 토큰 어휘 크기 |
| hidden_size | 7,168 | 히든 차원 |
| num_hidden_layers | 61 | 디코더 레이어 수 |
| num_attention_heads | 128 | 어텐션 헤드 수 |
| kv_lora_rank | 512 | KV 압축 차원 |
| n_routed_experts | 256 | 라우팅 전문가 수 |
| num_experts_per_tok | 8 | 토큰당 활성화 전문가 |

In [2]:
@dataclass
class DeepSeekV3Config:
    # 기본 설정
    vocab_size: int = 102400
    hidden_size: int = 7168
    num_hidden_layers: int = 61
    num_attention_heads: int = 128
    
    # MLA 설정
    kv_lora_rank: int = 512
    q_lora_rank: int = 1536
    qk_nope_head_dim: int = 128
    qk_rope_head_dim: int = 64
    v_head_dim: int = 128
    
    # MoE 설정
    n_routed_experts: int = 256
    n_shared_experts: int = 2
    num_experts_per_tok: int = 8
    moe_intermediate_size: int = 2048
    
    # 기타
    intermediate_size: int = 18432
    max_position_embeddings: int = 163840
    first_k_dense_replace: int = 3

config = DeepSeekV3Config()

print("📊 DeepSeek V3 설정")
print("=" * 50)
for field, value in vars(config).items():
    print(f"{field:>25}: {value:>10,}" if isinstance(value, int) else f"{field:>25}: {value}")

📊 DeepSeek V3 설정
               vocab_size:    102,400
              hidden_size:      7,168
        num_hidden_layers:         61
      num_attention_heads:        128
             kv_lora_rank:        512
              q_lora_rank:      1,536
         qk_nope_head_dim:        128
         qk_rope_head_dim:         64
               v_head_dim:        128
         n_routed_experts:        256
         n_shared_experts:          2
      num_experts_per_tok:          8
    moe_intermediate_size:      2,048
        intermediate_size:     18,432
  max_position_embeddings:    163,840
    first_k_dense_replace:          3


## 2. 전체 아키텍처 시각화

In [3]:
print("""
┌─────────────────────────────────────────────────────────────┐
│                   DeepSeek V3 Architecture                  │
│               총 671B 파라미터 / 토큰당 37B 활성화            │
└─────────────────────────────────────────────────────────────┘

                      Input Tokens
                          │
                          ▼
                  ┌───────────────┐
                  │Token Embedding│
                  └───────┬───────┘
                          │
    ┌─────────────────────┼─────────────────────┐
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder Layer 0-2             │ │
    │  │           (Dense MLP)                │ │
    │  │  RMSNorm → MLA → Add Residual        │ │
    │  │  RMSNorm → MLP → Add Residual        │ │
    │  └──────────────────────────────────────┘ │
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder Layer 3-60            │ │
    │  │            (MoE)                     │ │
    │  │  RMSNorm → MLA → Add Residual        │ │
    │  │  RMSNorm → MoE → Add Residual        │ │
    │  │  (256 experts, 8 active per token)   │ │
    │  └──────────────────────────────────────┘ │
    │                     │                     │
    │               × 61 layers                 │
    └─────────────────────┼─────────────────────┘
                          │
                  ┌───────────────┐
                  │ Final RMSNorm │
                  └───────┬───────┘
                          │
                  ┌───────────────┐
                  │   LM Head     │
                  └───────┬───────┘
                          │
                          ▼
                    Output Logits
""")

print("\n💡 핵심 포인트:")
print("   - 처음 3개 레이어는 Dense MLP")
print("   - 나머지 58개 레이어는 MoE")
print("   - 모든 레이어에서 MLA 사용")


┌─────────────────────────────────────────────────────────────┐
│                   DeepSeek V3 Architecture                  │
│               총 671B 파라미터 / 토큰당 37B 활성화            │
└─────────────────────────────────────────────────────────────┘

                      Input Tokens
                          │
                          ▼
                  ┌───────────────┐
                  │Token Embedding│
                  └───────┬───────┘
                          │
    ┌─────────────────────┼─────────────────────┐
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder Layer 0-2             │ │
    │  │           (Dense MLP)                │ │
    │  │  RMSNorm → MLA → Add Residual        │ │
    │  │  RMSNorm → MLP → Add Residual        │ │
    │  └──────────────────────────────────────┘ │
    │                     │                     │
    │  ┌──────────────────────────────────────┐ │
    │  │        Decoder 

## 📝 이해도 테스트 1: 파라미터 계산

In [4]:
# 파라미터 수 계산

# Embedding
embed_params = config.vocab_size * config.hidden_size

# Dense MLP (per layer)
dense_mlp_params = 3 * config.hidden_size * config.intermediate_size

# MoE (per layer)
expert_params = 3 * config.hidden_size * config.moe_intermediate_size
moe_params = (config.n_routed_experts * expert_params + 
              config.n_shared_experts * expert_params +
              config.n_routed_experts * config.hidden_size)  # router

# LM Head
lm_head_params = config.hidden_size * config.vocab_size

print("📊 파라미터 분석")
print("=" * 60)
print(f"Token Embedding:           {embed_params:>15,}")
print(f"Dense MLP (per layer):     {dense_mlp_params:>15,}")
print(f"MoE (per layer):           {moe_params:>15,}")
print(f"LM Head:                   {lm_head_params:>15,}")

# 총계 추정 (간소화)
dense_layers = config.first_k_dense_replace
moe_layers = config.num_hidden_layers - dense_layers

total_estimate = (embed_params + 
                  dense_layers * dense_mlp_params +
                  moe_layers * moe_params +
                  lm_head_params)

print(f"\n총 파라미터 추정치: {total_estimate/1e9:.1f}B")
print("(실제값: 671B - MLA 파라미터 등 미포함)")

📊 파라미터 분석
Token Embedding:               734,003,200
Dense MLP (per layer):         396,361,728
MoE (per layer):            11,364,204,544
LM Head:                       734,003,200

총 파라미터 추정치: 661.8B
(실제값: 671B - MLA 파라미터 등 미포함)


## 3. 추론 흐름

### Autoregressive Generation 과정

**Step 1: Prefill Phase (입력 처리)**
- 전체 입력 시퀀스를 한 번에 처리
- KV 캐시 저장

**Step 2: Decode Phase (토큰 생성)**
- 새 토큰 하나씩 생성
- KV 캐시 활용으로 효율적 처리

In [5]:
print("""
🔄 Autoregressive Generation 과정
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Step 1: Prefill Phase
─────────────────────

   Input: "DeepSeek V3는"
   
   ┌────────────────────────────────────────────────────────┐
   │ 1. Tokenization                                        │
   │    "DeepSeek V3는" → [token_1, token_2, token_3, ...]  │
   │                                                        │
   │ 2. Forward through all layers                          │
   │    - MLA: 모든 토큰 간 어텐션                            │
   │    - KV Cache 저장 (c_KV, k_rope)                      │
   │    - MoE: 각 토큰별 Top-8 전문가                        │
   │                                                        │
   │ 3. LM Head → Sampling → next_token                     │
   └────────────────────────────────────────────────────────┘

Step 2: Decode Phase (반복)
─────────────────────────

   ┌────────────────────────────────────────────────────────┐
   │ 1. 새 토큰 임베딩                                       │
   │                                                        │
   │ 2. Forward with KV Cache                               │
   │    - Query: 새 토큰만 계산                              │
   │    - Key, Value: 캐시에서 가져옴                        │
   │                                                        │
   │ 3. LM Head → Sampling → next_token                     │
   │                                                        │
   │ 4. KV Cache 업데이트                                   │
   └────────────────────────────────────────────────────────┘
   
   → EOS 토큰이 나올 때까지 반복

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("💡 핵심: KV 캐시 덕분에 Decode Phase가 효율적!")


🔄 Autoregressive Generation 과정
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Step 1: Prefill Phase
─────────────────────

   Input: "DeepSeek V3는"
   
   ┌────────────────────────────────────────────────────────┐
   │ 1. Tokenization                                        │
   │    "DeepSeek V3는" → [token_1, token_2, token_3, ...]  │
   │                                                        │
   │ 2. Forward through all layers                          │
   │    - MLA: 모든 토큰 간 어텐션                            │
   │    - KV Cache 저장 (c_KV, k_rope)                      │
   │    - MoE: 각 토큰별 Top-8 전문가                        │
   │                                                        │
   │ 3. LM Head → Sampling → next_token                     │
   └────────────────────────────────────────────────────────┘

Step 2: Decode Phase (반복)
─────────────────────────

   ┌────────────────────────────────────────────────────────┐
   │ 1. 새 토큰 임베딩                               

## 4. Multi-Token Prediction (MTP)

### 개념
- 기존: 한 번에 하나의 다음 토큰 예측
- MTP: 한 번에 여러 개의 토큰 예측

### 장점
1. 학습 효율성 향상
2. Speculative Decoding으로 추론 가속화

In [6]:
print("""
📊 Multi-Token Prediction (MTP)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

기존 Next-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris]  (1개)

Multi-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris, ., It, is]  (4개 동시!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Speculative Decoding 과정:

  Step 1: Draft (MTP 헤드로 여러 토큰 예측)
          → [Paris, ., It, is]
  
  Step 2: Verify (메인 모델로 검증)
          → [Paris ✅, . ✅, It ❌, ...]
  
  Step 3: Accept
          → "Paris ." 출력 (한 번에 2개 토큰!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("💡 핵심: 예측 + 검증으로 추론 속도 향상!")


📊 Multi-Token Prediction (MTP)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

기존 Next-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris]  (1개)

Multi-Token Prediction:
  Input: "The capital of France is"
  예측: [Paris, ., It, is]  (4개 동시!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Speculative Decoding 과정:

  Step 1: Draft (MTP 헤드로 여러 토큰 예측)
          → [Paris, ., It, is]
  
  Step 2: Verify (메인 모델로 검증)
          → [Paris ✅, . ✅, It ❌, ...]
  
  Step 3: Accept
          → "Paris ." 출력 (한 번에 2개 토큰!)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 핵심: 예측 + 검증으로 추론 속도 향상!


## 5. 혁신 요약

In [ ]:
print("""
🏆 DeepSeek V3 주요 혁신 요약
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1️⃣ Multi-head Latent Attention (MLA)
   ─────────────────────────────────
   핵심: KV 캐시를 저차원 잠재 벡터로 압축
   
   수식: c_KV = W_DKV × h
         k = [W_UK × c_KV; RoPE(W_KR × h)]
   
   효과: ✓ KV 캐시 메모리 73% 절감
         ✓ 긴 컨텍스트 (160K) 처리 가능

2️⃣ DeepSeekMoE with Auxiliary-Loss-Free Load Balancing
   ─────────────────────────────────────────────────────
   핵심: Bias term으로 로드 밸런싱
   
   수식: s'_i = s_i + b_i
         y = Shared(h) + Σ gᵢ × Expertᵢ(h)
   
   효과: ✓ 256개 전문가 중 8개만 활성화
         ✓ 671B 파라미터, 37B 활성화

3️⃣ Multi-Token Prediction (MTP)
   ─────────────────────────────
   핵심: 한 번에 여러 토큰 예측
   
   효과: ✓ 학습 효율성 향상
         ✓ Speculative Decoding으로 추론 가속화

4️⃣ FP8 Mixed Precision Training
   ─────────────────────────────
   핵심: FP8 정밀도로 학습
   
   효과: ✓ 학습 비용 약 $5.6M (매우 저렴!)
         ✓ 성능 저하 없음

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("📊 성능: 오픈소스 모델 중 최고 성능, GPT-4o/Claude 3.5와 경쟁")

## 📝 최종 퀴즈

In [ ]:
final_quiz = {
    "Q1: DeepSeek V3의 총 파라미터는 671B이다": None,
    "Q2: 토큰당 활성화 파라미터는 37B이다": None,
    "Q3: 모든 61개 레이어가 MoE를 사용한다": None,
    "Q4: MLA는 KV 캐시 메모리를 절감한다": None,
    "Q5: MTP는 추론 속도를 향상시킨다": None,
}

# 여기에 답을 입력하세요 (True or False)

In [ ]:
final_answers = {
    "Q1: DeepSeek V3의 총 파라미터는 671B이다": True,
    "Q2: 토큰당 활성화 파라미터는 37B이다": True,
    "Q3: 모든 61개 레이어가 MoE를 사용한다": False,  # 처음 3개는 Dense
    "Q4: MLA는 KV 캐시 메모리를 절감한다": True,
    "Q5: MTP는 추론 속도를 향상시킨다": True,
}

print("📋 최종 퀴즈 정답")
print("=" * 60)
correct = 0
for q, a in final_answers.items():
    user_ans = final_quiz.get(q)
    is_correct = user_ans == a
    if is_correct:
        correct += 1
    status = "✅" if is_correct else "❌"
    print(f"{status} {q}")
    print(f"   정답: {a}, 당신의 답: {user_ans}\n")

print(f"\n🎯 최종 점수: {correct}/{len(final_answers)} ({correct/len(final_answers)*100:.0f}%)")

## 🎉 학습 완료!

축하합니다! DeepSeek V3 아키텍처 학습을 완료했습니다.

### 📚 학습 내용 요약

1. **01_overview** - 전체 아키텍처 개요
2. **02_rmsnorm_rope** - RMSNorm과 RoPE
3. **03_mla_attention** - Multi-head Latent Attention
4. **04_moe_routing** - DeepSeekMoE와 라우팅
5. **05_full_model** - 전체 모델 구조와 추론 흐름

### 📖 추가 학습 자료

- 논문: https://arxiv.org/pdf/2412.19437
- 코드: https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v3/
- 모델: https://huggingface.co/deepseek-ai/DeepSeek-V3